<a href="https://colab.research.google.com/github/i-am-styles/En-fr-translator/blob/main/LLM_set4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print(f"Pytorch version: {torch.__version__}")
print(f"Device: {device}")

if torch.cuda.is_available():
  print(f"Device name: {torch.cuda.get_device_name(0)}")

Pytorch version: 2.11.0+cu130
Device: cuda:0
Device name: Tesla T4


In [2]:
from datasets import load_dataset
dataset = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr",
    split="train",

)

print(f"Total sentence pairs: {len(dataset)}")


README.md:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

en-fr/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

en-fr/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/127085 [00:00<?, ? examples/s]

Total sentence pairs: 127085


In [3]:
dataset = dataset.shuffle(seed=42).select(range(20000))
print(f"Total sentence pairs: {len(dataset)}")

Total sentence pairs: 20000


In [4]:
english_sentences = [example["translation"]["en"] for example in dataset]
french_sentences = [example["translation"]["fr"] for example in dataset]


print(english_sentences[0])
print(french_sentences[0])

"Not if it were my own brother!" cried d’Artagnan, as if carried away by his enthusiasm.
«Non, fût-ce mon frère!» s'écria d'Artagnan comme emporté par l'enthousiasme.


In [5]:
clean_pairs = [
    (english_sentence.strip(), french_sentence.strip())
    for english_sentence, french_sentence in zip(english_sentences, french_sentences)
    if english_sentence.strip() and french_sentence.strip()
]

english_sentences = [en for en,fr in clean_pairs]
french_sentences = [fr for en,fr in clean_pairs]



In [6]:
from sklearn.model_selection import train_test_split

train_en, val_en, train_fr, val_fr = train_test_split(english_sentences, french_sentences,random_state=42, test_size=0.2)
print(len(train_fr))
print(len(train_en))

16000
16000


In [7]:
from tokenizers import Tokenizer,models,pre_tokenizers,trainers

def make_Tokens(sentences):
  tokenizer = Tokenizer(
      models.BPE(unk_token="[UNK]")
  )
  tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
  trainer = trainers.BpeTrainer(
      vocab_size=10000,
      min_frequency=20,
      special_tokens=["[UNK]","[CLS]","[SEP]","[PAD]","[MASK]"]
  )
  tokenizer.train_from_iterator(sentences,trainer=trainer)

  return tokenizer

In [8]:
english_tokenizer = make_Tokens(english_sentences)
french_tokenizer = make_Tokens(french_sentences)

In [9]:
from tokenizers.processors import TemplateProcessing

def add_special_tokens(tokenizer):
  cls_id = tokenizer.token_to_id("[CLS]")
  sep_id = tokenizer.token_to_id("[SEP]")

  tokenizer.post_processor = TemplateProcessing(
      single=f"[CLS] $A [SEP]",
      special_tokens=[
          ("[CLS]",cls_id),
          ("[SEP]",sep_id)
      ]
  )


In [10]:
add_special_tokens(english_tokenizer)
add_special_tokens(french_tokenizer)

encoded = english_tokenizer.encode("I love machine learning")

print(encoded.tokens)
print(encoded.ids)

['[CLS]', 'I', 'love', 'mach', 'ine', 'lear', 'ning', '[SEP]']
[1, 39, 731, 3122, 314, 1826, 520, 2]


In [11]:
english_ids = [english_tokenizer.encode(en).ids for en in train_en]
french_ids = [french_tokenizer.encode(fr).ids for fr in train_fr]

print(english_ids[0])
print(french_ids[0])

[1, 39, 731, 169, 610, 2]
[1, 276, 206, 1400, 15, 91, 2]


In [12]:
english_length = [len(en) for en in english_ids]
french_length = [len(fr) for fr in french_ids]

print(max(english_length))
print(max(french_length))

337
322


In [13]:
def pad_sequence(sequences,pad_id,max_length):
  padded_sequences = []
  for seq in sequences:
    padding_length = max_length - len(seq)
    padded_seq = seq + [pad_id] * padding_length
    padded_sequences.append(padded_seq)
  return padded_sequences

In [14]:
english_pad_id = english_tokenizer.token_to_id("[PAD]")
french_pad_id = french_tokenizer.token_to_id("[PAD]")

english_max_length = max(english_length)
french_max_length = max(french_length)

english_padded_ids = pad_sequence(english_ids,english_pad_id,english_max_length)
french_padded_ids = pad_sequence(french_ids,french_pad_id,french_max_length)

In [15]:
def create_attention_mask(padded_sequences,pad_id):
  masks = []

  for seq in padded_sequences:
    mask = []

    for token_id in seq:
      if token_id == pad_id:
        mask.append(0)
      else:
        mask.append(1)
    masks.append(mask)
  return masks

In [16]:
english_mask = create_attention_mask(english_padded_ids,english_pad_id)
french_mask = create_attention_mask(french_padded_ids,french_pad_id)

In [17]:
english_padded_tensor = torch.tensor(english_padded_ids)
french_padded_tensor = torch.tensor(french_padded_ids)
english_mask_tensor = torch.tensor(english_mask)
french_mask_tensor = torch.tensor(french_mask)

In [18]:
print(english_padded_tensor.shape)
print(french_padded_tensor.shape)
print(english_mask_tensor.shape)
print(french_mask_tensor.shape)

print(english_padded_tensor.dtype)
print(french_padded_tensor.dtype)
print(english_mask_tensor.dtype)
print(french_mask_tensor.dtype)

print(english_pad_id)
print(french_pad_id)

torch.Size([16000, 337])
torch.Size([16000, 322])
torch.Size([16000, 337])
torch.Size([16000, 322])
torch.int64
torch.int64
torch.int64
torch.int64
3
3


In [19]:
val_english_ids = [english_tokenizer.encode(en).ids for en in val_en]
val_french_ids = [french_tokenizer.encode(fr).ids for fr in val_fr]

val_eng__length = [len(ids) for ids in val_english_ids]
val_fr__length = [len(ids) for ids in val_french_ids]

val_eng_max_length = max(val_eng__length)
val_fr_max_length = max(val_fr__length)

print(val_eng_max_length)
print(val_fr_max_length)

259
289


In [20]:
val_english_padded = pad_sequence(val_english_ids,english_pad_id,val_eng_max_length)
val_french_padded = pad_sequence(val_french_ids,french_pad_id,val_fr_max_length)

val_english_mask = create_attention_mask(val_english_padded,english_pad_id)
val_french_mask = create_attention_mask(val_french_padded,french_pad_id)

val_eng_tensor = torch.tensor(val_english_padded,dtype=torch.long)
val_fr_tensor = torch.tensor(val_french_padded,dtype=torch.long)

val_eng_mask_tensor = torch.tensor(val_english_mask,dtype=torch.long)
val_fr_mask_tensor = torch.tensor(val_french_mask,dtype=torch.long)

print(val_eng_tensor.shape)
print(val_fr_tensor.shape)
print(val_eng_mask_tensor.shape)
print(val_fr_mask_tensor.shape)

torch.Size([4000, 259])
torch.Size([4000, 289])
torch.Size([4000, 259])
torch.Size([4000, 289])


In [21]:
from torch.utils.data import TensorDataset,DataLoader

train_dataset = TensorDataset(
    english_padded_tensor,
    french_padded_tensor,
    english_mask_tensor,
    french_mask_tensor
)

val_dataset = TensorDataset(
    val_eng_tensor,
    val_fr_tensor,
    val_eng_mask_tensor,
    val_fr_mask_tensor
)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader = DataLoader(val_dataset,batch_size=32,shuffle=False)

batch = next(iter(train_loader))
print(batch)

[tensor([[   1,    6,  362,  ...,    3,    3,    3],
        [   1,  202, 3507,  ...,    3,    3,    3],
        [   1,  537, 4011,  ...,    3,    3,    3],
        ...,
        [   1,   39,  753,  ...,    3,    3,    3],
        [   1,    6,  797,  ...,    3,    3,    3],
        [   1,  703,  121,  ...,    3,    3,    3]]), tensor([[   1,  209,  358,  ...,    3,    3,    3],
        [   1,  342, 1424,  ...,    3,    3,    3],
        [   1,  495, 4431,  ...,    3,    3,    3],
        ...,
        [   1,  276,  289,  ...,    3,    3,    3],
        [   1,  124, 2753,  ...,    3,    3,    3],
        [   1,  495, 2996,  ...,    3,    3,    3]]), tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1,

In [22]:
class EncoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.attention = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )
        self.norm3 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, key_padding_mask):
        attn_out, _ = self.attention(x, x, x, key_padding_mask=key_padding_mask, need_weights=False)
        x = self.norm2(x + self.dropout(attn_out))
        x = self.norm3(x + self.dropout(self.ffn(x)))
        return x


class encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, num_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_embedding = nn.Embedding(block_size, embedding_dim)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.1)
        self.layers = nn.ModuleList([
            EncoderBlock(embedding_dim) for _ in range(num_layers)
        ])

    def forward(self, x, attention_mask):
        key_padding_mask = attention_mask == 0
        x = self.embedding(x)
        pos_ids = torch.arange(x.size(1), device=x.device)
        x = x + self.positional_embedding(pos_ids).unsqueeze(0)
        x = self.dropout(self.norm1(x))
        for layer in self.layers:
            x = layer(x, key_padding_mask)
        return x

In [23]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [24]:
vocab_size = english_tokenizer.get_vocab_size()
embedding_dim = 512
block_size = english_max_length

eng_sentences = english_padded_tensor[0:16].to(device)
eng_mask = english_mask_tensor[0:16].to(device)

print(eng_sentences.shape)
print(eng_mask.shape)

encoder_model = encoder(vocab_size,embedding_dim,block_size,num_layers=4).to(device)
output = encoder_model(eng_sentences,eng_mask)
print(output.shape)

torch.Size([16, 337])
torch.Size([16, 337])
torch.Size([16, 337, 512])


In [25]:
decoder_vocab_size = french_tokenizer.get_vocab_size()
decoder_embedding_dim = 512
decoder_block_size = french_max_length

print(decoder_vocab_size)
print(decoder_embedding_dim)
print(decoder_block_size)

4873
512
322


In [26]:
class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads=8):
        super().__init__()
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.self_attn = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(embedding_dim)
        self.cross_attn = nn.MultiheadAttention(embedding_dim, num_heads, batch_first=True)
        self.norm3 = nn.LayerNorm(embedding_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embedding_dim, 4 * embedding_dim),
            nn.ReLU(),
            nn.Linear(4 * embedding_dim, embedding_dim)
        )
        self.norm4 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, encoder_output, causal_mask, decoder_pad_mask, encoder_pad_mask):
        a1, _ = self.self_attn(
            x, x, x,
            key_padding_mask=decoder_pad_mask,
            attn_mask=causal_mask,
            need_weights=False
        )
        x = self.norm2(x + self.dropout(a1))

        a2, _ = self.cross_attn(
            x, encoder_output, encoder_output,
            key_padding_mask=encoder_pad_mask,
            need_weights=False
        )
        x = self.norm3(x + self.dropout(a2))
        x = self.norm4(x + self.dropout(self.ffn(x)))
        return x


class decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, block_size, num_layers=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.positional_embedding = nn.Embedding(block_size, embedding_dim)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.dropout = nn.Dropout(0.1)
        self.layers = nn.ModuleList([
            DecoderBlock(embedding_dim) for _ in range(num_layers)
        ])
        self.norm_final = nn.LayerNorm(embedding_dim)
        self.output_layer = nn.Linear(embedding_dim, vocab_size)

    def forward(self, x, encoder_output, decoder_attention_mask, encoder_attention_mask):
        decoder_pad_mask = decoder_attention_mask == 0
        encoder_pad_mask = encoder_attention_mask == 0

        seq_len = x.size(1)
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device),
            diagonal=1
        )

        x = self.embedding(x)
        pos_ids = torch.arange(seq_len, device=x.device)
        x = x + self.positional_embedding(pos_ids).unsqueeze(0)
        x = self.dropout(self.norm1(x))

        for layer in self.layers:
            x = layer(
                x, encoder_output, causal_mask,
                decoder_pad_mask, encoder_pad_mask
            )

        x = self.norm_final(x)
        return self.output_layer(x)


In [27]:
decoder_model = decoder(decoder_vocab_size,decoder_embedding_dim,decoder_block_size,num_layers=4).to(device)

fr_sentences = french_padded_tensor[0:16].to(device)
fr_mask = french_mask_tensor[0:16].to(device)

print(fr_sentences.shape)
print(fr_mask.shape)
decoder_output = decoder_model(fr_sentences,output,fr_mask,eng_mask)
print(decoder_output.shape)

torch.Size([16, 322])
torch.Size([16, 322])
torch.Size([16, 322, 4873])


In [28]:
class Transformer(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, src_mask, tgt_mask):
        encoder_output = self.encoder(src, src_mask)

        logits = self.decoder(
            tgt,
            encoder_output,
            tgt_mask,
            src_mask
        )

        return logits

In [29]:
model = Transformer(
    encoder_model,
    decoder_model
).to(device)

print("Transformer ready")

Transformer ready


In [30]:
loss_fn = nn.CrossEntropyLoss(ignore_index=french_pad_id)
optimizer = optim.Adam(model.parameters(),lr=0.0001)

In [ ]:
epochs = 10

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for eng_batch,fr_batch,eng_mask,fr_mask in train_loader:
    eng_batch = eng_batch.to(device)
    fr_batch = fr_batch.to(device)
    eng_mask = eng_mask.to(device)
    fr_mask = fr_mask.to(device)

    decoder_input = fr_batch[:,:-1]
    decoder_target = fr_batch[:,1:]

    fr_mask = fr_mask[:,:-1]


    optimizer.zero_grad()

    logits = model(eng_batch,decoder_input,eng_mask,fr_mask)

    loss = loss_fn(
        logits.reshape(-1,logits.shape[-1]),
        decoder_target.reshape(-1)
    )



    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
    optimizer.step()

    total_loss += loss.item()

  avg_loss = total_loss / len(train_loader)


  print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f}")


Epoch [1/10] | Loss: 5.9297
Epoch [2/10] | Loss: 5.1678
Epoch [3/10] | Loss: 4.7777
Epoch [4/10] | Loss: 4.4684
Epoch [5/10] | Loss: 4.2003
Epoch [6/10] | Loss: 3.9548
Epoch [7/10] | Loss: 3.7291
Epoch [8/10] | Loss: 3.5143
Epoch [9/10] | Loss: 3.3099


In [ ]:
print("French CLS ID:", french_tokenizer.token_to_id("[CLS]"))
print("French SEP ID:", french_tokenizer.token_to_id("[SEP]"))

In [ ]:

import torch

model.eval()

french_vocab = french_tokenizer.get_vocab()

print("French special tokens:")
print([token for token in french_vocab if "CLS" in token.upper() or "SEP" in token.upper()])

french_cls_id = french_tokenizer.token_to_id("[CLS]")
french_sep_id = french_tokenizer.token_to_id("[SEP]")

if french_cls_id is None or french_sep_id is None:
    raise ValueError("French CLS/SEP token names match nahi hue. Upar printed token names check karo.")

def translate(sentence, max_length=50):
    tokens = english_tokenizer.encode(sentence).ids

    src = torch.tensor(
        [tokens], dtype=torch.long, device=device
    )

    src_mask = torch.ones_like(src)
    generated = [french_cls_id]

    with torch.no_grad():
        encoder_output = model.encoder(src, src_mask)

        for _ in range(max_length):
            tgt = torch.tensor(
                [generated], dtype=torch.long, device=device
            )

            tgt_mask = torch.ones_like(tgt)

            logits = model.decoder(
                tgt, encoder_output, tgt_mask, src_mask
            )

            next_token = logits[0, -1].argmax(dim=-1).item()

            if next_token == french_sep_id:
                break

            generated.append(next_token)

    return french_tokenizer.decode(generated[1:])


In [ ]:
sentences = [
    "Hello",
    "How are you?",
    "Good morning",
    "Thank you",
    "I love you"
]

for sentence in sentences:
    print("English:", sentence)
    print("French:", translate(sentence))
    print("-" * 30)